# Paper 3 — W3 DEV ladder (Colab runner)

Evaluates each NeuralCQR change on **DEV only**, per the spec
`docs/superpowers/specs/2026-09-17-paper3-audit-remediation-design.md`.

**Rules this notebook enforces**
- Everything is scored on DEV (2014–2015). TEST years (2019–2023) and the held-out
  LOSO states are never read here (D1).
- A change is accepted only if it improves DEV RMSE averaged over 5 seeds (D2).
- One variable changes per variant (D3).
- LightGBM gets the same DEV-based budget selection as NeuralCQR, so a win against
  an untuned comparator is not reported as a win.

Set the runtime to GPU: *Runtime → Change runtime type → T4 GPU*.


## 1. Project into place

Same layout as the master runner: a folder containing `code/` and `master_dataset/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Paper3_MASTER'   # must contain code/ and master_dataset/

import os
print('exists:', os.path.isdir(PROJECT_ROOT))
print(sorted(os.listdir(PROJECT_ROOT))[:10])

## 2. Pre-flight

Fails here rather than an hour into the run.

In [ ]:
import os, sys, importlib, torch

os.environ['PAPER3_DATA_SOURCE'] = 'master'
CODE_DIR = os.path.join(PROJECT_ROOT, 'code')
assert os.path.isdir(CODE_DIR), f'code/ not found at {CODE_DIR}'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, CODE_DIR)

import config; importlib.reload(config)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (this will be slow)')
print('data source:', config.DATA_SOURCE)

# The F0 flags must all be at their defaults: the ladder sets them per variant.
for flag in ('STANDARDIZE_TARGET', 'USE_EMA_WEIGHTS', 'NEURAL_MONOTONE_HEADS', 'LR_SCHEDULE_MATCH_DEV'):
    assert getattr(config, flag) is False, f'{flag} must start False'
print('F0 flags at defaults: OK')

## 3. Verify the code changes before trusting any number

Checks that the default path is bit-identical to the committed version, that
standardised training round-trips, that monotone heads cannot cross, and that the
EMA refit really returns averaged weights. Takes under a minute.

In [ ]:
!python -u code/diagnostics_loso/verify_f0_changes.py 2>&1 | tail -20

## 4. Run the ladder (5 seeds)

Variants: E0 baseline, E1 target standardisation + Huber delta, E2/E2b EMA refit vs
last-epoch refit, E3/E3b learning-rate and capacity, E4 monotone heads, E5
`lambda_width = 0`, LG0/LG1 LightGBM untuned vs DEV-tuned.

Expect roughly 1.5–3 hours on a T4. Output streams live.

**Before running:** make sure the `code/` folder on Drive is the updated one — `config.py` (F0 flags), `model_training.py`, `aci_calibrator.py`, `evaluation.py`, `utils.py`, `main.py`, `claim_consistency_audit.py` and `code/diagnostics_loso/`. The pre-flight cell fails loudly if the F0 flags are missing.

In [ ]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, '-u', 'code/diagnostics_loso/w3_dev_ladder.py', '--seeds', '5'],
    # Serial on purpose: --parallel exists for CPU machines, where a small net
    # cannot use more than ~1 core. On a GPU one process keeps the device busy.
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env=dict(os.environ, PAPER3_DATA_SOURCE='master'))

for line in proc.stdout:
    if any(k in line for k in ('---', 'seed ', 'DEV ', 'variant', 'written:', 'Traceback',
                               'Error', 'ERROR', '===', 'FIT n=')):
        print(line.rstrip())
print('exit code:', proc.wait())

## 5. Decision table (D2)

Accept a variant only if it improves DEV RMSE against E0.

In [ ]:
import pandas as pd, os

R = os.path.join(PROJECT_ROOT, 'outputs_diagnostics', 'reports')
s = pd.read_csv(os.path.join(R, 'w3_dev_ladder_summary.csv'))
raw = pd.read_csv(os.path.join(R, 'w3_dev_ladder_raw.csv'))

base = float(s.loc[s.variant == 'E0', 'rmse_mean'].iloc[0])
s['delta_rmse_vs_E0'] = s.rmse_mean - base
s['decision'] = ['ACCEPT' if d < 0 else 'REVERT' for d in s.delta_rmse_vs_E0]
s.loc[s.variant == 'E0', 'decision'] = 'baseline'

print(s[['variant', 'rmse_mean', 'rmse_sd', 'r2_mean', 'picp_mean', 'mpiw_mean',
         'crossing_mean', 'delta_rmse_vs_E0', 'decision']].to_string(index=False))

lgb = raw[raw.seed < 0]
if len(lgb):
    print('
LightGBM on the same DEV rows:')
    print(lgb[['variant', 'rmse', 'r2', 'picp', 'mpiw', 'best_epoch']].to_string(index=False))
    print('
D4 check: the backbone is whichever model has the lower DEV RMSE.')
    print('  best NeuralCQR variant :', f"{s.rmse_mean.min():.4f}")
    print('  best LightGBM          :', f"{lgb.rmse.min():.4f}")

## 6. Save back to Drive

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
archive = f'/content/paper3_w3_dev_ladder_{stamp}'
shutil.make_archive(archive, 'zip', os.path.join(PROJECT_ROOT, 'outputs_diagnostics', 'reports'))
print('wrote', archive + '.zip')